In [84]:
#from tabpfn import TabPFNClassifier
from tabpfn_extensions import TabPFNClassifier, interpretability
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from tqdm import tqdm
import json

import pandas as pd
from sklearn.impute import SimpleImputer


In [ ]:
def binarize(s):
    print(s.value_counts())
    counts = s.value_counts()
    minority_class = counts.idxmin()
    majority_class = counts.idxmax()
    binary_series = s.map({minority_class: 1, majority_class: 0})
    return binary_series


def split_data(X, y, test_size, random=False):
    if random:
        return train_test_split(X, y, stratify=y, test_size=test_size)
    else:

        pos_idxs = []
        for idx, y_ in enumerate(list(y)):
            if y_ == 1:
                pos_idxs += [idx]

        n = int(len(pos_idxs)*(1-test_size))
        cut_idx = pos_idxs[n]
                
        # Test set: only rows with indices in `test_pos_indices`
        X_test = X.iloc[cut_idx:]
        y_test = y.iloc[cut_idx:]

        # Train set: all other rows
        X_train = X.iloc[:cut_idx]
        y_train = y.iloc[:cut_idx]
    
        return X_train, X_test, y_train, y_test


def cv_tabpfn(df, test_size=0.33, random=True, impute=False):
    columns = list(df.columns)
    df = df[df[columns[0]].notnull()]
    y = binarize(df[columns[0]])
    x = df[columns[1:]]
    # x = pd.get_dummies(x)
    x_train, x_test, y_train, y_test = split_data(x, y, test_size, random)
    if impute:
        imputer = SimpleImputer()
        imputer.fit(x_train)
        x_train = imputer.transform(x_train)
        x_test = imputer.transform(x_test)
    mdl = TabPFNClassifier(ignore_pretraining_limits=True)
    mdl.fit(x_train, y_train)
    y_hat = mdl.predict_proba(x_test)[:,1]
    fpr, tpr, _ = roc_curve(y_test, y_hat)
    data = {"fpr": [float(x) for x in fpr], "tpr": [float(x) for x in tpr], "auroc": auc(fpr, tpr)}
    print(data["auroc"])
    return data


dataset_names = ["ml_d1_predelivery", "ml_d2_earlydeath", "ml_d3_latedeath"]

for dn in dataset_names:
    df = pd.read_csv("../data/processed/{0}.csv".format(dn))
    cv_data = []
    for _ in tqdm(range(10)):
        cv_data += [cv_tabpfn(df, random=True)]
    with open("../results/cv_{0}.json".format(dn), "w") as f:
        json.dump(cv_data, f, indent=4)

  0%|          | 0/10 [00:00<?, ?it/s]

Outcome Death
Live birth                   10422
Miscarriage or Stillbirth      174
Name: count, dtype: int64


 10%|█         | 1/10 [00:01<00:10,  1.15s/it]

0.7716467768257853
Outcome Death
Live birth                   10422
Miscarriage or Stillbirth      174
Name: count, dtype: int64


 20%|██        | 2/10 [00:02<00:09,  1.13s/it]

...

 90%|█████████ | 9/10 [00:10<00:01,  1.13s/it]

0.7972154222766218
Outcome Death
Live birth                   10422
Miscarriage or Stillbirth      174
Name: count, dtype: int64


100%|██████████| 10/10 [00:11<00:00,  1.13s/it]

0.7843023255813953


TypeError: Object of type ndarray is not JSON serializable

Early Neonatal Death
Alive    10297
Dead       112
Name: count, dtype: int64

In [15]:
mdl = TabPFNClassifier(ignore_pretraining_limits=True)
mdl.fit(df, df["Pre-term Delivery"])

/home/mduranfrigola/miniconda3/envs/tabpfn/lib/python3.12/site-packages/tabpfn/classifier.py:421: UserWarning: Number of samples 10422 is greater than the maximum Number of samples 10000 supported by the model. You may see degraded performance.
  X, y, feature_names_in, n_features_in = validate_Xy_fit(


TabPFNClassifier(ignore_pretraining_limits=True)

In [11]:
df

,Pre-term Delivery,Miscarriage,Outcome Death,Maternal Age,School Level,Years of Education,Parity,Maternal Height,Maternal Weight,Multiple Birth
0,Term,No miscarriage,Live birth,28.0,3=School,7.0,2.0,167.0,87.0,0.0
1,Term,No miscarriage,Live birth,19.0,3=School,7.0,0.0,154.0,66.0,0.0
2,Term,No miscarriage,Live birth,32.0,3=School,7.0,2.0,NaN,67.0,0.0
3,Term,No miscarriage,Live birth,32.0,3=School,9.0,3.0,154.0,97.0,0.0
4,Term,No miscarriage,Live birth,17.0,3=School,6.0,1.0,153.0,71.0,0.0
...,...,...,...,...,...,...,...,...,...,...
10591,Term,No miscarriage,Live birth,22.0,3=School,11.0,0.0,166.0,NaN,0.0
10592,Preterm,No miscarriage,Live birth,22.0,3=School,10.0,0.0,154.0,NaN,0.0
10593,Preterm,No miscarriage,Live birth,36.0,3=School,9.0,3.0,154.0,NaN,0.0
10594,Preterm,No miscarriage,Live birth,19.0,3=School,7.0,0.0,NaN,55.0,0.0
